# Golf Swing Data Pipeline & Feature Engineering Exploration

This notebook prototypes the parsing of the GolfDB dataset annotations (`golfDB.mat`), maps the 8 key swing events to relative frames in the local video files, runs the coordinate extraction pipeline, and engineers the **sliding window** temporal features for high-movement joints.

## Objectives:
1. Parse annotations from `data/golfDB.mat`.
2. Batch process golf swing videos using the Day 1 `GolfVideoProcessor`.
3. Apply a sliding window of $\pm 5$ frames for wrists, elbows, shoulders, and hips coordinates.
4. Label frames: exact milestone frames mapped to `1-8`, transitional frames to `0`.
5. Verify coordinate changes around the **Top of Backswing** milestone.
6. Create a visual display helper for Jupyter to inspect any row in the dataset.

In [1]:
import os
import sys
import scipy.io
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

# Add project root to sys.path
PROJECT_ROOT = '/Users/sagar/Documents/ML/golf-analysis'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data_processor import GolfVideoProcessor
from src.visualizer import draw_skeleton
from src.feature_engineer import engineer_sliding_window, HIGH_MOVEMENT_JOINTS

### Step 1: Parse the GolfDB Mat File

We load `data/golfDB.mat` using `scipy.io.loadmat` and construct a mapping of `video_id` to its 8 milestone indices. Since the `.mat` file contains absolute frame numbers relative to the original YouTube videos, we subtract the start frame (`events[0]`) to make them relative to our trimmed local videos.

In [2]:
mat_path = os.path.join(PROJECT_ROOT, 'data/golfDB.mat')

def parse_golfdb_mat(mat_path):
    mat_data = scipy.io.loadmat(mat_path)
    db = mat_data['golfDB'][0]
    
    metadata = []
    for i in range(len(db)):
        rec = db[i]
        vid_id = int(rec['id'][0][0])
        events = rec['events'][0]
        
        # Calculate relative event indices
        start_frame = events[0]
        rel_events = events - start_frame
        milestones = rel_events[1:9]
        
        # Extract additional metadata fields
        player = str(rec['player'][0]) if len(rec['player']) > 0 else ''
        sex = str(rec['sex'][0]) if len(rec['sex']) > 0 else ''
        club = str(rec['club'][0]) if len(rec['club']) > 0 else ''
        view = str(rec['view'][0]) if len(rec['view']) > 0 else ''
        slow = int(rec['slow'][0][0]) if len(rec['slow']) > 0 else 0
        bbox = rec['bbox'][0].tolist() if len(rec['bbox']) > 0 else []
        
        metadata.append({
            'video_id': vid_id,
            'player': player,
            'sex': sex,
            'club': club,
            'view': view,
            'slow': slow,
            'start_frame': start_frame,
            'end_frame': events[9],
            'bbox': bbox,
            'milestones': milestones.tolist()
        })
        
    return pd.DataFrame(metadata)

df_meta = parse_golfdb_mat(mat_path)
print(f'Loaded metadata for {len(df_meta)} videos.')
df_meta.head()

Loaded metadata for 1400 videos.


,video_id,player,sex,club,view,slow,start_frame,end_frame,bbox,milestones
0,0,SANDRA GAL,f,driver,down-the-line,0,408,545,"[0.09765625000000001, 0.006944444444444444, 0....","[47, 65, 68, 82, 87, 90, 93, 106]"
1,1,SANDRA GAL,f,driver,down-the-line,1,814,1137,"[0.039062500000000014, 0.0006944444444444445, ...","[40, 103, 117, 174, 192, 205, 216, 269]"
2,2,CHRIS DIMARCO,m,driver,down-the-line,0,521,745,"[0.165625, 0.0006944444444444445, 0.48359375, ...","[138, 157, 162, 171, 175, 177, 180, 194]"
3,3,CHRIS DIMARCO,m,driver,down-the-line,1,1106,1449,"[0.18515625, 0.0006944444444444445, 0.465625, ...","[84, 138, 158, 194, 207, 218, 229, 283]"
4,4,BROOKE HENDERSON,f,driver,down-the-line,0,157,250,"[0.11015625, 0.0006944444444444445, 0.4984375,...","[13, 26, 31, 40, 44, 48, 50, 63]"


### Validate parsed GolfDB labels

Before processing, we run a validation check on all annotations in `df_meta` to ensure they are chronologically ordered, within video boundaries, and conform to biomechanical speed limits (e.g. downswing takes between 3 and 35 frames).

In [3]:
def validate_and_filter_annotations(df_meta):
    print(f"Scanning annotations for {len(df_meta)} videos...\n")
    
    anomalies = []
    invalid_ids = set()
    
    for idx, row in df_meta.iterrows():
        vid_id = row['video_id']
        milestones = row['milestones'] # List of 8 indices
        end_frame = row['end_frame']
        start_frame = row['start_frame']
        video_length = end_frame - start_frame
        
        # 1. Check Monotonicity (strict chronological order)
        is_monotonic = all(x < y for x, y in zip(milestones, milestones[1:]))
        if not is_monotonic:
            anomalies.append({
                'video_id': vid_id,
                'issue': 'Out of chronological order',
                'details': f"Milestones: {milestones}"
            })
            invalid_ids.add(vid_id)
            continue
            
        # 2. Check Boundary Limits
        out_of_bounds = any(m < 0 or m > video_length for m in milestones)
        if out_of_bounds:
            anomalies.append({
                'video_id': vid_id,
                'issue': 'Milestones out of video bounds',
                'details': f"Milestones: {milestones}, Video Length: {video_length}"
            })
            invalid_ids.add(vid_id)
            continue

    # Summary Report
    df_anomalies = pd.DataFrame(anomalies)
    if df_anomalies.empty:
        print("✅ Validation complete: 0 anomalies found! All labels are structurally sound and safe for training.")
        return df_meta
    else:
        print(f"⚠️ Validation complete: Found {len(df_anomalies)} anomalies out of {len(df_meta)} videos.")
        display(df_anomalies)
        
        # Filter df_meta to exclude invalid video IDs
        df_meta_clean = df_meta[~df_meta['video_id'].isin(invalid_ids)].reset_index(drop=True)
        print(f"🧹 Filtered out {len(invalid_ids)} invalid videos. New metadata count: {len(df_meta_clean)}")
        return df_meta_clean

# Validate and filter df_meta
df_meta = validate_and_filter_annotations(df_meta)


Scanning annotations for 1400 videos...

⚠️ Validation complete: Found 1 anomalies out of 1400 videos.


,video_id,issue,details
0,677,Out of chronological order,"Milestones: [92, 114, 131, 146, 158, 168, 167,..."


🧹 Filtered out 1 invalid videos. New metadata count: 1399


### Step 2: Batch Process Video Landmarks

We loop through a subset of local videos (configured by `NUM_VIDEOS_TO_PROCESS`), process each video using `GolfVideoProcessor`, and stack the resulting DataFrames. This returns raw, smoothed, and normalized landmark coordinate streams.

In [ ]:
# NUM_VIDEOS_TO_PROCESS is configurable. Use 10 for testing/prototyping.
# Set to None (or len(df_meta)) to process the entire dataset.
NUM_VIDEOS_TO_PROCESS = len(df_meta)
VIDEO_DIR = os.path.join(PROJECT_ROOT, 'data/videos_160/videos_160')
TEMP_DIR = os.path.join(PROJECT_ROOT, 'data/processed/temp_features')
os.makedirs(TEMP_DIR, exist_ok=True)

from tqdm.notebook import tqdm

processed_dfs = []
skipped_count = 0
processed_count = 0

# Set up the visual progress bar
pbar = tqdm(range(NUM_VIDEOS_TO_PROCESS), desc="Processing Videos")

for i in pbar:
    video_id = df_meta.iloc[i]['video_id']
    video_path = os.path.join(VIDEO_DIR, f'{video_id}.mp4')
    cache_path = os.path.join(TEMP_DIR, f'{video_id}.csv')
    
    # 1. Check if we already processed this video (Resume Capability)
    if os.path.exists(cache_path):
        skipped_count += 1
        pbar.set_postfix(video_id=video_id, status="Cached")
        df_vid = pd.read_csv(cache_path)
        processed_dfs.append(df_vid)
        continue
        
    # 2. Check if video file exists
    if not os.path.exists(video_path):
        pbar.set_postfix(video_id=video_id, status="Missing")
        continue
        
    # 3. Process video (Instantiating inside loop to avoid MediaPipe timestamp issues)
    pbar.set_postfix(video_id=video_id, status="Processing")
    try:
        processor = GolfVideoProcessor()
        df_vid = processor.process_video(video_path, video_id=video_id)
        # Save checkpoint immediately
        df_vid.to_csv(cache_path, index=False)
        processed_dfs.append(df_vid)
        processed_count += 1
        processor.close()
    except Exception as e:
        print(f"\nError processing video {video_id}: {e}")

print(f"\nProcessing finished. Total processed: {processed_count}, Loaded from cache: {skipped_count}")

# Concatenate all processed videos
df_raw_features = pd.concat(processed_dfs, ignore_index=True)
print(f'Combined features shape: {df_raw_features.shape}')


Starting processing for first 1399 videos...


I0000 00:00:1783214573.351147 5056352 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1783214573.416530 5056352 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1783214573.456583 5056357 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783214573.470527 5056357 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783214573.502559 5056362 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


[1/1399] Video 0 - Processing landmarks...
[2/1399] Video 1 - Processing landmarks...
Error processing video 1: Input timestamp must be monotonically increasing.
[3/1399] Video 2 - Processing landmarks...
Error processing video 2: Input timestamp must be monotonically increasing.
[4/1399] Video 3 - Processing landmarks...
Error processing video 3: Input timestamp must be monotonically increasing.
[5/1399] Video 4 - Processing landmarks...
Error processing video 4: Input timestamp must be monotonically increasing.
[6/1399] Video 5 - Processing landmarks...
Error processing video 5: Input timestamp must be monotonically increasing.
[7/1399] Video 6 - Processing landmarks...
Error processing video 6: Input timestamp must be monotonically increasing.
[8/1399] Video 7 - Processing landmarks...
Error processing video 7: Input timestamp must be monotonically increasing.
[9/1399] Video 8 - Processing landmarks...
Error processing video 8: Input timestamp must be monotonically increasing.
[10/1

[ERROR:0@12.958] global cap_ffmpeg_impl.hpp:1471 open Could not open codec mpeg4, error: -35
[ERROR:0@12.958] global cap_ffmpeg_impl.hpp:1479 open VIDEOIO/FFMPEG: Failed to initialize VideoCapture


### Step 3: Engineer Sliding Window Features

For each frame $T$, we append the coordinates from $T-5$ and $T+5$ as new columns. We group by `video_id` during shift operations to prevent temporal bleed across different videos. 

Boundary frames ($T < 5$ or $T > N - 6$) are padded using backfill and forward-fill within their respective video groups to keep columns free of `NaN` values without losing milestone rows near boundaries.

In [ ]:
df_windowed = engineer_sliding_window(df_raw_features, HIGH_MOVEMENT_JOINTS)
print(f'Windowed features shape: {df_windowed.shape}')

### Step 4: Label Frames

We assign labels `1-8` to the exact milestone frames matching the metadata indices, and `0` to all other transitional frames.

In [ ]:
def assign_labels(df, df_meta):
    df_labeled = df.copy()
    
    # Create mapping of video_id to its milestone list
    meta_dict = df_meta.set_index('video_id')['milestones'].to_dict()
    
    # Initialize label column to 0
    df_labeled['label'] = 0
    
    # Iterate through groups and assign labels
    for vid_id, group in df_labeled.groupby('video_id'):
        if vid_id not in meta_dict:
            continue
        milestones = meta_dict[vid_id]
        
        # Assign labels 1-8 to the milestone frame indices
        for milestone_idx, frame_idx in enumerate(milestones):
            label_val = milestone_idx + 1
            
            # Match the exact frame in the video
            mask = (df_labeled['video_id'] == vid_id) & (df_labeled['frame_index'] == frame_idx)
            df_labeled.loc[mask, 'label'] = label_val
            
    # Merge video-level metadata columns into the frame-level DataFrame
    metadata_cols = ['video_id', 'player', 'sex', 'club', 'view', 'slow']
    df_labeled = df_labeled.merge(df_meta[metadata_cols], on='video_id', how='left')
            
    return df_labeled

df_final = assign_labels(df_windowed, df_meta)
print('Label distribution:')
print(df_final['label'].value_counts())

### Step 5: Save Master Dataset

In [ ]:
processed_dir = os.path.join(PROJECT_ROOT, 'data/processed')
os.makedirs(processed_dir, exist_ok=True)
master_csv_path = os.path.join(processed_dir, 'master_training_dataset.csv')

df_final.to_csv(master_csv_path, index=False)
print(f'Master training dataset saved to {master_csv_path}')

### Step 6: Visual & Biomechanical Verification

Let's check the wrist coordinates around the **Top of Backswing** (label 4).

**MediaPipe Coordinate System Notes**:
- Y increases downwards. Therefore, a higher joint position on-screen has a **smaller Y coordinate** value.
- At the Top of Backswing ($T$), the wrist is at its peak elevation (minimum Y coordinate value).
- In the backswing ($T-5$) and downswing ($T+5$), the wrist should be physically lower (larger Y coordinate values).

Let's print the actual values to verify this physical relationship.

### Step 7: Notebook Frame Display Function

Here we define `show_frame_with_milestone(row)` using our shared `draw_skeleton` visualization utility to view processed skeleton overlays directly in the notebook.

In [ ]:
def show_frame_with_milestone(row, video_dir=VIDEO_DIR):
    video_id = int(row['video_id'])
    frame_idx = int(row['frame_index'])
    label = int(row['label'])
    
    video_path = os.path.join(video_dir, f'{video_id}.mp4')
    
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print(f'Failed to read frame {frame_idx} from {video_path}')
        return
        
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Draw skeleton overlay using the imported shared draw_skeleton
    draw_skeleton(frame_rgb, row, prefix='smooth_', line_color=(0, 255, 0), joint_color=(255, 0, 0))
    
    event_names = {
        0: 'Transition',
        1: 'Address',
        2: 'Toe-up (Backswing)',
        3: 'Mid-backswing',
        4: 'Top of Backswing',
        5: 'Mid-downswing',
        6: 'Impact',
        7: 'Mid-follow-through',
        8: 'Finish'
    }
    
    label_text = f'Video {video_id} | Frame {frame_idx} - {event_names.get(label, "Unknown")}'
    
    # Draw text background and text
    cv2.rectangle(frame_rgb, (10, 10), (320, 45), (0, 0, 0), -1)
    cv2.putText(frame_rgb, label_text, (15, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    
    img = Image.fromarray(frame_rgb)
    scale = 2
    display(img.resize((img.width * scale, img.height * scale), Image.Resampling.LANCZOS))


# Find a top of backswing frame and show it
top_row = df_final[df_final['label'] == 4].iloc[0]
show_frame_with_milestone(top_row)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from ipywidgets import interact, IntSlider

# Define video ID and path at the top level
video_id = 1
video_path = os.path.join(VIDEO_DIR, f"{video_id}.mp4")

# Load total frames
cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# Load df_final directly from the processed CSV to avoid dependencies on previous cells
csv_path = os.path.join(PROJECT_ROOT, 'data/processed/master_training_dataset.csv')
if os.path.exists(csv_path):
    df_final_data = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(f"Processed master dataset CSV not found at {csv_path}. Please run the cells above first to generate it.")

# Filter dataframe for this video once to make lookups fast
df_video = df_final_data[df_final_data['video_id'] == video_id]

# Swing milestone names
event_names = {
    0: 'Transition', 1: 'Address', 2: 'Toe-up (Backswing)', 3: 'Mid-backswing',
    4: 'Top of Backswing', 5: 'Mid-downswing', 6: 'Impact', 7: 'Mid-follow-through', 8: 'Finish'
}

# Helper to fetch a frame, draw skeleton, and add top-padded text overlays
def get_inspected_frame(f_idx, title, y_val):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        frame_rgb = np.zeros((240, 320, 3), dtype=np.uint8) # Fallback black frame
    else:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 1. Draw skeleton on the native 240x320 frame first
        row_data = df_video[df_video['frame_index'] == f_idx]
        if not row_data.empty:
            draw_skeleton(frame_rgb, row_data.iloc[0], prefix='smooth_', line_color=(0, 255, 0), joint_color=(255, 0, 0))
            
    # 2. Add 55px black border at the top so text NEVER covers the golfer
    padded = cv2.copyMakeBorder(frame_rgb, 55, 0, 0, 0, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    
    # 3. Draw larger text inside the safe padded area
    cv2.putText(padded, title, (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    y_str = f"Wrist Y: {y_val:.4f}" if y_val is not None and not np.isnan(y_val) else "Wrist Y: N/A"
    cv2.putText(padded, y_str, (10, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
    
    return padded

# Comparison function
def show_comparison(frame):
    if df_video.empty:
        print(f"No pipeline data available for Video {video_id} in the CSV.")
        return
        
    # Lookup the row for the current frame T
    row_t = df_video[df_video['frame_index'] == frame]
    if row_t.empty:
        print(f"No pipeline data available for Frame {frame}")
        return
        
    row_t = row_t.iloc[0]
    label = int(row_t['label'])
    
    # Extract the sliding window wrist coordinates
    y_prev = row_t.get('norm_right_wrist_y_t-5', np.nan)
    y_t = row_t.get('norm_right_wrist_y', np.nan)
    y_next = row_t.get('norm_right_wrist_y_t+5', np.nan)
    
    # Check if Y_t is a local peak (minimum coordinate value)
    is_peak = (y_prev > y_t) and (y_next > y_t)
    
    # Generate the 3 frames side-by-side with padding
    img_prev = get_inspected_frame(frame - 5, f"T-5 (Frame {frame - 5})", y_prev)
    img_t = get_inspected_frame(frame, f"T (Frame {frame}) - {event_names.get(label, 'Unknown')}", y_t)
    img_next = get_inspected_frame(frame + 5, f"T+5 (Frame {frame + 5})", y_next)
    
    # Concatenate horizontally
    combined = np.hstack((img_prev, img_t, img_next))
    
    # Render (Increased scaling to 1.5x for larger, clearer view)
    img_pil = Image.fromarray(combined)
    display(img_pil.resize((int(img_pil.width * 1.5), int(img_pil.height * 1.5)), Image.Resampling.LANCZOS))
    
    # Display the comparison status below the image
    print(f"Current Frame T: {frame} | Label: {label} ({event_names.get(label, 'Unknown')})")
    print(f"Condition Check: (Y_prev > Y_t) and (Y_next > Y_t)")
    print(f"                 ({y_prev:.4f} > {y_t:.4f}) and ({y_next:.4f} > {y_t:.4f}) -> {is_peak}")

# Direct call to prevent double-rendering bug
interact(show_comparison, frame=IntSlider(min=5, max=total_frames-6, step=1, value=82))


### Step 8: Raw Video Slider (Inspect Unprocessed Videos)

This helper allows you to inspect any raw video frame-by-frame, including ones that have not been processed by the pipeline yet (e.g. video `1381`).

In [ ]:
import os
import cv2
from PIL import Image
from ipywidgets import interact, IntSlider

# Define raw video ID and path at the top level
raw_video_id = 1381
raw_video_path = os.path.join(VIDEO_DIR, f"{raw_video_id}.mp4")

# Load total frames
cap = cv2.VideoCapture(raw_video_path)
total_frames = int(cap.get(cap.get(cv2.CAP_PROP_FRAME_COUNT) if cap.isOpened() else 0))
# Let's do it safely
if cap.isOpened():
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
else:
    total_frames = 100
cap.release()

@interact(frame=IntSlider(min=0, max=total_frames-1, step=1, value=82))
def show_raw_frame(frame):
    cap = cv2.VideoCapture(raw_video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
    ret, img_frame = cap.read()
    cap.release()
    
    if ret:
        img_rgb = cv2.cvtColor(img_frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img_rgb)
        # Display the frame scaled up 2x
        display(img.resize((img.width * 2, img.height * 2), Image.Resampling.LANCZOS))
        print(f"Video {raw_video_id} | Showing Frame: {frame} / {total_frames}")
